# Content2Form - TLN
### Authors: _Simone Multari_, _Mattia Mondino_, _Loris Signoretti_
Il seguente progetto mira a sviluppare un classificatore in grado di assegnare un valore di `BASICNESS` alle parole. Questa misura riflette il grado di complessità di una parola, rappresentando quanto una parola sia comune e di facile comprensione.

## Risorse Utilizzate

- **Dataset di Training e Valutazione**  
  Il [dataset](https://github.com/federicotorrielli/stableKnowledge/tree/master/json_analyzer) fornisce una lista di parole annotate con il relativo synset e un valore di basicness: `middle` o `advanced`.

- **WordNet**  
  Utilizzato come risorsa lessicale per estrarre caratteristiche (features) linguistiche utili alla classificazione.

- **CMU Pronouncing Dictionary**  
  [CMU Pronouncing Dictionary](https://www.nltk.org/_modules/nltk/corpus/reader/cmudict.html) è stato impiegato per calcolare la difficoltà di pronuncia delle parole, basandosi sul numero e la complessità dei fonemi.


- **Corpus di Storie per Bambini**  
  [Dataset di storie](https://huggingface.co/datasets/lilithyu/kaggle-child-stories) per bambini, impiegato per analizzare la frequenza delle parole.


## Modello di Classificazione

Per il training è stato utilizzato un **Random Forest Classifier**, addestrato sulle caratteristiche estratte per prevedere il livello di basicness delle parole.

In [11]:
import pandas as pd

from C2F_functions import get_genus, find_senses
import spacy
import nltk
from sentence_transformers import SentenceTransformer


nltk.download("wordnet")
nlp = spacy.load("en_core_web_md")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kurai\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [12]:
model = SentenceTransformer('all-MiniLM-L6-v2')
dataset = pd.read_csv("TLN-definitions-24.csv", sep=",")
pd.set_option('display.max_colwidth', 300)
dataset = dataset.iloc[:, 1:]
# map on dataset for coumn and use fillna to fill in missing values
dataset = dataset.map(lambda x: x if pd.notnull(x) else "")
dataset

,pen,cigarette,cloud,ontology
0,is an object used to write on paper filled with ink,is an object filled mainly bu tobacco and smoked for pleasure,is an atmospheric phenomenon,"is the philosophical study of the nature and structure of beings and reality, their categories and the relationship between them"
1,an object that is used for write in a peper,is a cylinder of paper with tobacco inside,is a specific atmosphere condition,a field of philosophy used of universal universal
2,A pen is a tool that is used to write into a paper using ink,A cigarette is an object that is smoked for pleasure,A cloud is a white sky object that is responsible for rain,A ontology is a tool used to create a a common knowledge between different communicating agents
3,A pen is a object used to write with ink.,A cigarette is an object that can contains tobacco. It is used for smoking.,"A cloud is a mass of a water drops in the atmosphere, witch can take different form. It can produce a meteorological phenomenom, such as rain, snow or hail storm.",An ontology is a method to represent knowledge.
4,"An object that can be used to write on papers, it is usually a tool that permit to spread ink on paper following the direction of the hand of the utilizer; it can be stylographic, in this case it is required to be manually inked, or ball pen, in this case the ink is flowing from the body of the ...","A product made of tobacco inserted in a paper; could be made industrially or hand-rolled using papers; it can contain or not a filter, usually made of cotton. The cigarette is then light up and smoked by the user, who get a sensation of pleasure from the inhalation of nicotine.","The formations in the sky derived from the condensation of water particles in the atmosphere. They can be white or black, and usually this signals the possibility of an imminent rain.","The science that study the being, that is, the carachteristics related with the existence of some being. It is a philosophical concept, often studied in metaphysics, but the concept of ontology can be exploited in several other domains, such as medicine."
5,"A stick that can write down with ink some\r\nword, sketches and drawings",A consumable stick that can be \r\nlight up in order to inhalate it's \r\nsmoke. Usually filled with nicotine.,An abstraction of a set of services delivered\r\nby the internet.,The study of a formal definition of sense and being.
6,A pen is an object used for write on paper with ink,a cigarette is a cylinder with tobacco inside,a cloud isa cluster of tiny water drops,ontology is a branch of phylosophy
7,A pen is a tool used to write text,A cigarete is tabacco wrapped in paper used for smoking,a cloud is,
8,A pen is a writing tool using ink,A cigarete is tabacco wrapped in paper used for smoking,the cloud is a system of computer accessible from different users in different places,An ontology is a knowledge structure used to store relationships between entities
9,"A pen is a tool for writing on paper. Usually, it is black, blue or red.",A cigarette is a tool that allows you to smoke tobacco,the term cloud refers to a system of remote server,Ontology is a branch of Philosophy.


### Pen

In [13]:
pen_definition_list = dataset["pen"].tolist()
pen_genus = get_genus(pen_definition_list, 3, "pen", nlp)
print("Genus: ", pen_genus)

pen_symilarity_score = find_senses(pen_genus, pen_definition_list, model, max_iter = 100, min_score=0.5, transformer_weight=1, overlap_weight=0)

    # create DataFrame to visualize symilarity scores and add synset.description as third column
pen_symilarity_score_pd = pd.DataFrame(pen_symilarity_score, columns=["Synset", "Score"])
pen_symilarity_score_pd["Description"] = pen_symilarity_score_pd["Synset"].map(lambda x: x.definition())
pen_symilarity_score_pd

Genus:  ['write', 'paper', 'ink']


,Synset,Score,Description
0,Synset('ink.n.01'),0.805955,a liquid used for printing or writing or drawing
1,Synset('sheet.n.02'),0.803977,paper used for writing or printing
2,Synset('writing_paper.n.01'),0.787880,paper material made into thin sheets that are sized to take ink; used for writing correspondence and manuscripts
3,Synset('india_ink.n.01'),0.780506,a black liquid ink used for printing or writing or drawing
4,Synset('drawing_paper.n.01'),0.755778,paper that is specially prepared for use in drafting
5,Synset('carbon_paper.n.01'),0.754072,a thin paper coated on one side with a dark waxy substance (often containing carbon); used to transfer characters from the original to an under sheet of paper
6,Synset('marking_ink.n.01'),0.753062,an indelible ink for marking clothes or linens etc.
7,Synset('ink.v.03'),0.746119,fill with ink
8,Synset('tracing_paper.n.01'),0.744129,a semitransparent paper that is used for tracing drawings
9,Synset('leaf.n.02'),0.740128,a sheet of any written or printed material (especially in a manuscript or book)


### Cigarette

In [14]:
cigarette_definition_list = dataset["cigarette"].tolist()
cigarette_genus = get_genus(cigarette_definition_list, 3, "cigarette", nlp)
print("Genus: ", cigarette_genus)

cigarette_symilarity_score = find_senses(cigarette_genus, cigarette_definition_list, model, max_iter = 100, min_score=0.5, transformer_weight=1, overlap_weight=0)

cigarette_symilarity_score_pd = pd.DataFrame(cigarette_symilarity_score, columns=["Synset", "Score"])
cigarette_symilarity_score_pd["Description"] = cigarette_symilarity_score_pd["Synset"].map(lambda x: x.definition())
cigarette_symilarity_score_pd

Genus:  ['tobacco', 'paper', 'object']


,Synset,Score,Description
0,Synset('cigar.n.01'),0.827272,a roll of tobacco for smoking
1,Synset('cigarillo.n.01'),0.817070,small cigar or cigarette wrapped in tobacco instead of paper
2,Synset('cigarette.n.01'),0.811732,finely ground tobacco wrapped in paper; for smoking
3,Synset('claro.n.01'),0.804954,a cigar made with light-colored tobacco
4,Synset('cubeb.n.03'),0.798227,a cigarette containing cubeb
5,Synset('cigarette_paper.n.01'),0.792661,a strong tissue paper that burns evenly and is sufficiently porous to control the burning of the tobacco in a cigarette
6,Synset('joint.n.06'),0.791321,marijuana leaves rolled into a cigarette for smoking
7,Synset('filter-tipped_cigarette.n.01'),0.787759,a cigarette with a filter tip
8,Synset('filler.n.05'),0.787043,the tobacco used to form the core of a cigar
9,Synset('smoking_mixture.n.01'),0.776211,a blend of tobaccos to be smoked in a pipe


### Cloud

In [15]:
cloud_definition_list = dataset["cloud"].tolist()
cloud_genus = get_genus(cloud_definition_list, 3, "cloud", nlp)
print("Genus: ", cloud_genus)

cloud_symilarity_score = find_senses(cloud_genus, cloud_definition_list, model, max_iter = 100, min_score=0.5, transformer_weight=1, overlap_weight=0)

cloud_symilarity_score_pd = pd.DataFrame(cloud_symilarity_score, columns=["Synset", "Score"])
cloud_symilarity_score_pd["Description"] = cloud_symilarity_score_pd["Synset"].map(lambda x: x.definition())
cloud_symilarity_score_pd

Genus:  ['water', 'atmosphere', 'sky']


,Synset,Score,Description
0,Synset('fog.n.02'),0.715718,an atmosphere in which visibility is reduced because of a cloud of some substance
1,Synset('rain.n.02'),0.689636,drops of fresh water that fall as precipitation from clouds
2,Synset('mackerel_sky.n.01'),0.686475,a sky filled with rows of cirrocumulus or small altocumulus clouds
3,Synset('genius_loci.n.01'),0.634996,the special atmosphere of a place
4,Synset('lake.n.01'),0.626593,a body of (usually fresh) water surrounded by land
5,Synset('condensate.n.01'),0.625253,a product of condensation
6,Synset('atmosphere.n.04'),0.624761,the weather or climate at some place
7,Synset('low.n.01'),0.623802,an air mass of lower pressure; often brings precipitation
8,Synset('sky.n.01'),0.618982,the atmosphere and outer space as viewed from the earth
9,Synset('atmosphere.n.05'),0.615673,the envelope of gases surrounding any celestial body


### Ontology

In [16]:
ontology_definition_list = dataset["ontology"].tolist()
ontology_genus = get_genus(ontology_definition_list, 3, "ontology", nlp)
print("Genus: ", ontology_genus)

ontology_symilarity_score = find_senses(ontology_genus, ontology_definition_list, model, max_iter = 100, min_score=0.5, transformer_weight=1, overlap_weight=0)

ontology_symilarity_score_pd = pd.DataFrame(ontology_symilarity_score, columns=["Synset", "Score"])
ontology_symilarity_score_pd["Description"] = ontology_symilarity_score_pd["Synset"].map(lambda x: x.definition())
ontology_symilarity_score_pd

Genus:  ['study', 'structure', 'knowledge']


,Synset,Score,Description
0,Synset('lexical_semantics.n.01'),0.682704,the branch of semantics that studies the meanings and relations of words
1,Synset('formal_semantics.n.01'),0.681118,the branch of semantics that studies the logical aspects of meaning
2,Synset('mathematics.n.01'),0.665429,a science (or group of related sciences) dealing with the logic of quantity and shape and arrangement
3,Synset('cognitive_semantics.n.01'),0.661661,the branch of semantics that studies the cognitive aspects of meaning
4,Synset('ontology.n.01'),0.660850,(computer science) a rigorous and exhaustive organization of some knowledge domain that is usually hierarchical and contains all the relevant entities and their relations
5,Synset('cosmography.n.01'),0.648442,the science that maps the general features of the universe; describes both heaven and earth (but without encroaching on geography or astronomy)
6,Synset('public_knowledge.n.01'),0.647223,knowledge that is available to anyone
7,Synset('structure.n.03'),0.646842,the complex composition of knowledge as elements and their combinations
8,Synset('ology.n.01'),0.645741,an informal word (abstracted from words with this ending) for some unidentified branch of knowledge
9,Synset('structure.n.03'),0.645473,the complex composition of knowledge as elements and their combinations
